# Лабораторная работа № 4 (в методичке — № 3). Часть 1: библиотека NumPy
### Дисциплина «Технологии программирования». Вариант 1

**Цель:** получить навыки векторных вычислений с использованием NumPy.

**Задание.** Пусть $\mathcal{N}(a,\sigma^2)$ — нормальное распределение, $\mathbb{I}[\beta(x)]$ — индикаторная функция,
$\odot$ — поэлементное произведение, $\operatorname{Tr}\mathbf{A}$ — след матрицы. Создать матрицы и векторы

$$
\mathbf{M}=\begin{pmatrix}3&2&2&\dots&2\\2&5&2&\dots&2\\2&2&7&\dots&2\\\vdots&&&\ddots&\vdots\\2&2&2&\dots&171\end{pmatrix}\in\mathbb{N}^{85\times85},\qquad
\mathbf{N}=\begin{pmatrix}4&4&5&4&4&5&4&\dots\\4&7&4&7&4&7&4&\dots\\4&4&5&4&4&5&4&\dots\\\vdots&&&&&&&\ddots\end{pmatrix}\in\mathbb{N}^{85\times85},
$$

$$
\mathbf{p}=\{1+p_i : p_i\sim\mathcal{N}(2,4)\}_{i=0}^{84},\qquad \mathbf{q}^T=\{\mathbf{N}_{24,j}\odot\mathbf{N}_{71,j}\}_{j=0}^{84}
$$

и вычислить (вариант 1):

$$
\mathbf{f}=\operatorname{Tr}(\mathbf{NM})\,\mathbf{p},\qquad g=\sum_{i=0}^{85}\mathbb{I}[i \bmod 2 = 0]\,(2\mathbf{p}\odot\mathbf{q})_i .
$$

**Циклы использовать запрещено** — все вычисления векторные.

**Технологии:** Python 3, NumPy, Jupyter Notebook / Google Colab.

In [1]:
import numpy as np

print('NumPy version:', np.__version__)
np.set_printoptions(linewidth=120, precision=4, suppress=True)

n = 85                 # размерность матриц и векторов
idx = np.arange(n)     # индексы 0..84

NumPy version: 2.5.5


## 1. Матрица M

Вне диагонали все элементы равны 2, на диагонали — нечетные числа $3, 5, 7, \dots$: $\mathbf{M}_{ii} = 3 + 2i$
(при $i = 84$ получаем $3 + 168 = 171$, что совпадает с условием). Создаем матрицу, заполненную двойками,
и записываем диагональ через `np.fill_diagonal`.

In [2]:
M = np.full((n, n), 2, dtype=np.int64)
np.fill_diagonal(M, 3 + 2 * idx)

print('shape:', M.shape, 'dtype:', M.dtype)
print(M[:5, :6])
print('M[84, 84] =', M[84, 84])

shape: (85, 85) dtype: int64
[[ 3  2  2  2  2  2]
 [ 2  5  2  2  2  2]
 [ 2  2  7  2  2  2]
 [ 2  2  2  9  2  2]
 [ 2  2  2  2 11  2]]
M[84, 84] = 171


## 2. Матрица N

Строки чередуются: четные строки (0, 2, 4, …) — периодическая последовательность `4 4 5`,
нечетные (1, 3, 5, …) — последовательность `4 7`. Строим два шаблона строк с помощью `np.resize`
(повторяет массив до нужной длины) и выбираем нужный шаблон для каждой строки через `np.where`
с broadcasting: маска четности имеет форму `(85, 1)`, шаблоны — `(85,)`.

In [3]:
even_row = np.resize([4, 4, 5], n)     # 4 4 5 4 4 5 ...
odd_row = np.resize([4, 7], n)         # 4 7 4 7 ...
N = np.where((idx % 2 == 0)[:, np.newaxis], even_row, odd_row)

print('shape:', N.shape, 'dtype:', N.dtype)
print(N[:5, :10])

shape: (85, 85) dtype: int64
[[4 4 5 4 4 5 4 4 5 4]
 [4 7 4 7 4 7 4 7 4 7]
 [4 4 5 4 4 5 4 4 5 4]
 [4 7 4 7 4 7 4 7 4 7]
 [4 4 5 4 4 5 4 4 5 4]]


## 3. Вектор p

$p_i \sim \mathcal{N}(2, 4)$: математическое ожидание $a = 2$, **дисперсия** $\sigma^2 = 4$, значит,
стандартное отклонение $\sigma = 2$ (именно его принимает `normal` в параметре `scale`).
Для воспроизводимости результата фиксируем зерно генератора.

In [4]:
rng = np.random.default_rng(seed=42)
p = 1 + rng.normal(loc=2, scale=2, size=n)

print('shape:', p.shape)
print(p[:10])
print(f'выборочное среднее = {p.mean():.3f} (теор. 1 + 2 = 3), '
      f'выборочная дисперсия = {p.var(ddof=1):.3f} (теор. 4)')

# Выборка из 85 элементов мала, поэтому выборочные оценки заметно отличаются от теоретических.
# Проверим параметризацию на большой выборке: дисперсия должна быть ≈ 4.
big = 1 + np.random.default_rng(0).normal(loc=2, scale=2, size=1_000_000)
print(f'контроль на 10^6 значений: среднее = {big.mean():.3f}, дисперсия = {big.var():.3f}')

shape: (85,)
[ 3.6094  0.92    4.5009  4.8811 -0.9021  0.3956  3.2557  2.3675  2.9664  1.2939]
выборочное среднее = 2.998 (теор. 1 + 2 = 3), выборочная дисперсия = 2.371 (теор. 4)
контроль на 10^6 значений: среднее = 3.002, дисперсия = 4.005


## 4. Вектор q

$\mathbf{q}^T$ — поэлементное произведение 24-й и 71-й строк матрицы $\mathbf{N}$.
Строка 24 — четная (`4 4 5 …`), строка 71 — нечетная (`4 7 …`).

In [5]:
q = N[24, :] * N[71, :]

print('shape:', q.shape)
print('N[24]:', N[24, :12])
print('N[71]:', N[71, :12])
print('q    :', q[:12])

shape: (85,)
N[24]: [4 4 5 4 4 5 4 4 5 4 4 5]
N[71]: [4 7 4 7 4 7 4 7 4 7 4 7]
q    : [16 28 20 28 16 35 16 28 20 28 16 35]


## 5. Вычисление f = Tr(NM) · p

Произведение матриц — оператор `@`, след — `np.trace`. След — скаляр, поэтому $\mathbf{f}$ — вектор той же
длины, что и $\mathbf{p}$.

Для проверки вычислим след другим способом без формирования произведения:
$\operatorname{Tr}(\mathbf{NM}) = \sum_{i,k}\mathbf{N}_{ik}\mathbf{M}_{ki} = \sum (\mathbf{N}\odot\mathbf{M}^T)$.

In [6]:
trace_NM = np.trace(N @ M)
f = trace_NM * p

print('Tr(NM) =', trace_NM)
print('Проверка: sum(N ⊙ Mᵀ) =', np.sum(N * M.T), '| einsum:', np.einsum('ij,ji->', N, M))
assert trace_NM == np.sum(N * M.T)

print('f.shape =', f.shape)
print('f =')
print(f)

Tr(NM) = 111564
Проверка: sum(N ⊙ Mᵀ) = 111564 | einsum: 111564
f.shape = (85,)
f =
[ 402682.9126  102642.4263  502138.6744  544558.324  -100638.5796   44139.291   363216.7735  264129.4229  330943.1913
  144354.0145  530910.3113  508239.159   349425.2975  586211.076   439006.4245  142959.7913  416970.625   120738.443
  530698.8588  323552.1313  293444.0305  182757.5526  607475.2038  300212.1457  239120.0697  256121.1451  453465.084
  416232.8032  426784.2022  430820.2288  812553.5459  244009.4302  220396.3043  153116.5187  472134.2566  586597.3297
  309267.1317  147229.5656  150727.1553  479857.4676  500532.8167  455884.9256  186198.15    386493.6917  360727.8712
  383487.5492  529132.1604  384582.4276  486176.6255  349770.7826  399202.6332  475550.0793    9559.7362  263364.4008
  229738.6904  192140.4635  273300.0598  668255.2649  141500.8348  550742.0127  -40803.3664  259969.773   371006.7659
  465494.6164  493386.5643  511709.9819  256881.6721  231528.3692  526130.4424  292006.6486 

## 6. Вычисление g = Σ I[i mod 2 = 0] · (2p ⊙ q)ᵢ

Индикатор $\mathbb{I}[i \bmod 2 = 0]$ оставляет только слагаемые с четными индексами.
Векторы имеют индексы $0..84$, поэтому верхний предел суммы $85$ фактически означает «до конца вектора»
(элемента с индексом 85 не существует).

Вычисляем двумя способами: через булеву маску-индикатор и через срез с шагом 2.

In [7]:
indicator = (idx % 2 == 0).astype(int)       # I[i mod 2 = 0]
g = np.sum(indicator * (2 * p * q))

g_slice = np.sum((2 * p * q)[::2])           # альтернативный способ: срез с шагом 2
assert np.isclose(g, g_slice)

print('Число слагаемых с четным индексом:', indicator.sum())
print(f'g = {g:.6f}')
print(f'g (срез [::2]) = {g_slice:.6f}')

Число слагаемых с четным индексом: 43
g = 4523.938321
g (срез [::2]) = 4523.938321


## 7. Итог

In [8]:
print('Tr(NM) =', trace_NM)
print('f = Tr(NM)·p, первые 5 элементов:', f[:5])
print(f'g = {g:.4f}')

Tr(NM) = 111564
f = Tr(NM)·p, первые 5 элементов: [ 402682.9126  102642.4263  502138.6744  544558.324  -100638.5796]
g = 4523.9383


## Выводы

1. Все матрицы и векторы построены **без циклов**: `np.full` + `np.fill_diagonal` для $\mathbf{M}$,
   периодические шаблоны `np.resize` и `np.where` с **broadcasting** (маска `(85, 1)` × шаблон `(85,)`) для $\mathbf{N}$,
   генератор `default_rng().normal` для $\mathbf{p}$, индексация строк и поэлементное умножение для $\mathbf{q}$.
2. При генерации $\mathbf{p}$ важно помнить, что в обозначении $\mathcal{N}(a, \sigma^2)$ второй параметр —
   дисперсия, а NumPy принимает стандартное отклонение: `scale = sqrt(4) = 2`.
3. $\operatorname{Tr}(\mathbf{NM}) = 111\,564$; результат подтвержден независимым расчетом
   $\sum(\mathbf{N}\odot\mathbf{M}^T)$, который не требует $O(n^3)$ умножения матриц.
   $\mathbf{f}$ — вектор из 85 элементов, пропорциональный $\mathbf{p}$.
4. Индикаторная функция реализуется булевой маской (`idx % 2 == 0`), что эквивалентно срезу `[::2]`;
   значение $g$ зависит от случайного вектора $\mathbf{p}$ (при `seed=42` $g \approx 4523.94$).
5. Векторные операции NumPy делают код коротким, близким к математической записи и значительно
   более быстрым, чем циклы Python, так как вычисления выполняются в скомпилированном коде.